**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/wang-2019"], check=True)
    os.chdir("/content/encore/entries/wang-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Wang et al. (2019) from SPARC dataset 212

**Paper.** Wang L, Morotti S, Tapa S, Francis Stuart SD, Jiang Y, Wang Z, Myles RC, Brack KE, Ng GA,
Bers DM, Grandi E, Ripplinger CM (2019). Different paths, same destination: divergent action
potential responses produce conserved cardiac fight-or-flight response in mouse and rabbit hearts.
*J Physiol* 597, 3867-3883. [doi:10.1113/jp278016](https://doi.org/10.1113/jp278016). The claims file
(`claims/wang-2019.yaml`) was written from the accepted author manuscript in PubMed Central
(PMC6675632), with Methods and figure legends; the paper has no supplement.

**Dataset.** [SPARC:212](https://sparc.science/datasets/212), "Sympathetic nerve stimulation of mouse
and rabbit hearts", version 1, [doi:10.26275/lok5-wje6](https://doi.org/10.26275/lok5-wje6), published
2022-01-05. Version 1 is the only published version, and the paper (2019) cites no version, so it is
the one pinned. Files are read from the public bucket `sparc-prod-aod-discover-publish50-use1` at the
object versions listed in that version's manifest.

**What the dataset holds.** No optical maps or traces. Its readme says: "There is no primary data,
only a derivative folder containing experimental optical mapping data summarized in the multi
spreadsheet excel table." The workbook `files/derivative/exp_data_Wang_et_al.xlsx` gives one value per
heart for Figure 1A and 1D (heart rate), Figure 2 (time courses of HR, APD80, CaTD80 and relative CaT
amplitude), Figure 3B to 3F, and Figures 5F and 7F (APD change at base and apex). The code folder holds
the MATLAB models used for Figure 8.

| Claim | Tested here |
|---|---|
| fig1a-rabbit-hr-increase, fig1d-mouse-hr-increase | Tested (Fig1 sheet) |
| fig1bc-rabbit-ap-cat-response, fig1ef-mouse-ap-cat-response | Tested on the first 20 s of the 60 s SNS series (Fig2 sheet); the Fig. 1 hearts' APD, CaTD and amplitude are not in the dataset |
| fig2a-rabbit-monotonic, fig2d-mouse-biphasic-apd, fig2cf-cat-amplitude-similar | Tested (Fig2 sheet) |
| fig3abc-tyramine-na-mimic-sns, fig3d-propranolol-abolishes, fig3e-rate-step-biphasic, fig3f-constant-pacing-sns-prolongs | Tested (Fig3 sheet) |
| fig4g-5f-rabbit-base-apex, fig7f-mouse-base-apex | Tested (Fig5 and Fig7 sheets) |
| fig4ab, fig4cd, fig4e, fig5c, fig6ab, fig6e, fig6fh, fig7c | Blocked, data missing: no maps and no paced 20 s series in the dataset |
| fig8e, fig8f, fig8ij, fig9 (simulations) | Blocked, not runnable: the models are in the dataset (`files/code`) but are MATLAB code, and neither MATLAB nor GNU Octave can be installed in the entry's locked Python environment (section "Simulation claims") |

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

import sns

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

man = sns.manifest()
files = sns.file_table(man)
print(f"SPARC:{sns.DATASET_ID} version {man['version']} ({man['@id']}), published {man['datePublished']}")
print("related publications:", man["relatedPublications"])
print(f"{len(files)} files, {files['size'].sum() / 1e6:.2f} MB")
print(files.sort_values("path").to_string(index=False))
tables = sns.load_tables(man)

## Data check

The SPARC metadata files are read first, then each table of the workbook is checked against the
counts and units the paper gives.

In [ ]:
for path in ["readme.md", "files/README.txt", "metadata/records/subject.csv", "metadata/records/sample.csv"]:
    print(f"--- {path}")
    print(sns.read_file(man, path).decode("utf8", "replace").strip())
desc = sns.read_excel_sheets(man, "files/dataset_description.xlsx")["Sheet1"]
print("--- files/dataset_description.xlsx (element, value)")
for _, row in desc[[0, 3]].dropna(how="all").iterrows():
    print(f"  {row[0]}: {str(row[3])[:120]}")
for path in sorted(p for p in files.path if p.endswith("manifest.xlsx")):
    m = sns.read_excel_sheets(man, path)["Sheet1"]
    print(f"--- {path}")
    print(m.iloc[1:, [0, 2]].to_string(index=False, header=False))

In [ ]:
# The archive lists 61 files for version 1 against the manifest's 58; the other files are under
# revisions/, listed here from the Pennsieve discover API.
import requests
def browse(path):
    r = requests.get(f"https://api.pennsieve.io/discover/datasets/{sns.DATASET_ID}/versions/{sns.VERSION}/files/browse",
                     params={"path": path, "limit": 100}, timeout=60)
    r.raise_for_status()
    return r.json()["files"]
info = requests.get(f"https://api.pennsieve.io/discover/datasets/{sns.DATASET_ID}/versions/{sns.VERSION}", timeout=60).json()
print("archive fileCount:", info["fileCount"], "; manifest files:", len(man["files"]))
for f in browse("revisions/1"):
    print(f"  {f['path']}  {f.get('size')} bytes  {f.get('fileType', f['type'])}")


The dataset declares 0 subjects and 0 samples (`dataset_description.xlsx`, and empty `subject.csv`
and `sample.csv`), and the curator's note calls it a computational modelling dataset. The workbook
holds measurements from hearts, but no table says which animal a column is, so hearts cannot be
followed across figures.

In [ ]:
sheets = tables["sheets"]
for name, sheet in sheets.items():
    print(f"=== sheet {name}: {sheet.shape[0]} rows x {sheet.shape[1]} columns; its text cells:")
    for (i, j), v in sheet.stack().items():
        if isinstance(v, str):
            print(f"  row {i + 1}, column {j + 1}: {v.strip()}")

In [ ]:
# The numeric blocks behind each table, as the workbook holds them (all columns, so that the
# heart columns, the empty no-SNS column of Figure 3F and the unlabelled column B can be seen).
def block(sheet, title, header):
    s = sheets[sheet]
    h = sns._find(s, header, sns._find(s, title))
    b = sns._numeric_block(s, h + 1)
    return b.loc[:, b.notna().any()]
print("=== Fig1, whole sheet"); print(sheets["Fig1"].fillna("").to_string())
for title, header in (("Effects of tyramine", "SNS Time"), ("Effects of Noradrenaline", "SNS Time"),
                      ("Effects of propranolol", "SNS Time"), ("APD in response to a step", "Time (sec)"),
                      ("APD over 60", "Time (sec)")):
    print(f"=== Fig3, '{title}' (columns numbered from 0, header row {sns._find(sheets['Fig3'], header, sns._find(sheets['Fig3'], title)) + 1})")
    print(sheets["Fig3"].iloc[sns._find(sheets["Fig3"], header, sns._find(sheets["Fig3"], title))].dropna().to_dict())
    print(block("Fig3", title, header).to_string())


In [ ]:
legend_n = {  # n in the paper's figure legends (claims file)
    "fig1_rabbit_hr": 5, "fig1_mouse_hr": 5, "fig2_rabbit": 4, "fig2_mouse": 5, "fig2_rabbit_cat": 6,
    "fig2_mouse_cat": 6, "fig3_tyramine": 4, "fig3_na": 4, "fig3_propranolol": 3, "fig3e": 3, "fig3f": 5,
    "fig5f_rabbit": 4, "fig7f_mouse": 5}
rows = []
for key, n in legend_n.items():
    tab = tables[key]
    if "variable" in tab:
        for var in tab.variable.unique():
            if var in ("unlabelled column B", "pacing rate (bpm)"):
                continue
            w = sns.wide(tab, var)
            rows.append({"table": key, "variable": var, "hearts": w.shape[1], "legend_n": n,
                         "times": ", ".join(f"{t:g}" for t in w.index), "missing": int(w.isna().sum().sum()),
                         "min": w.min().min(), "max": w.max().max()})
    else:
        cols = [c for c in tab.columns if c != "heart"]
        rows.append({"table": key, "variable": "/".join(cols), "hearts": len(tab), "legend_n": n, "times": "",
                     "missing": int(tab[cols].isna().sum().sum()), "min": tab[cols].min().min(),
                     "max": tab[cols].max().max()})
print(pd.DataFrame(rows).to_string(index=False))

Every table has the number of hearts its figure legend gives, except Figure 3F, whose "no SNS"
(baseline) series has 4 hearts while the SNS series and the legend have 5 (the second column of the
baseline block is empty). One noradrenaline heart has no value at 12.33 s. Ranges are consistent with
the stated units: heart rate in beats per minute (rabbit 122 to 273, mouse 224 to 572), APD80 and
CaTD80 in ms (rabbit 122 to 221, mouse 31 to 84), CaT amplitude relative to time 0 (0.97 to 1.20).

In [ ]:
# The four CSV files in files/derivative against the Fig2 sheet.
for species in ("mouse", "rabbit"):
    csv = pd.read_csv(pd.io.common.BytesIO(sns.read_file(
        man, f"files/derivative/Effects_of_SNS_on_heart_rate_APD_and_CaTD_in_{species}.csv")), header=0)
    k = csv.shape[1] // 3
    long = tables[f"fig2_{species}"]
    same = all(np.allclose(csv.iloc[:, 1 + i * k:1 + (i + 1) * k].values, sns.wide(long, var).T.values.T)
               for i, var in enumerate(["CaTD80", "APD80", "HR"]))
    print(f"{species} HR/APD/CaTD csv: {csv.shape[0]} times x {csv.shape[1] - 1} columns; identical to the Fig2 sheet: {same}")
    csv = pd.read_csv(pd.io.common.BytesIO(sns.read_file(
        man, f"files/derivative/Effects_of_SNS_on_CaT_Amplitude_in_{species}.csv")), header=0)
    same = np.allclose(csv.iloc[:, 1:].values, sns.wide(tables[f"fig2_{species}_cat"], "CaT amplitude").values)
    print(f"{species} CaT amplitude csv: identical to the Fig2 sheet: {same}")

# Any heart's series repeated within or across tables (the same recording deposited twice).
series = []
for key in ["fig2_rabbit", "fig2_mouse", "fig3_tyramine", "fig3_na", "fig3_propranolol", "fig3e", "fig3f",
            "fig2_rabbit_cat", "fig2_mouse_cat"]:
    tab = tables[key]
    for var in tab.variable.unique():
        if var in ("unlabelled column B", "pacing rate (bpm)"):
            continue
        w = sns.wide(tab, var)
        for h in w.columns:
            series.append((f"{key}/{var}/heart {h}", tuple(np.round(w[h].dropna().values, 3))))
dups = [(a[0], b[0]) for i, a in enumerate(series) for b in series[i + 1:] if a[1] == b[1]]
print("identical series:", dups or "none")
# Repeated adjacent values within a series.
for name, vals in series:
    rep = [i for i in range(1, len(vals)) if vals[i] == vals[i - 1]]
    if len(rep) and "propranolol" not in name and "cat" not in name.lower():
        print(f"{name}: value repeated at consecutive time points (index {rep}): {vals[:3]}")
fig3e = sns.wide(tables["fig3e"], "no SNS"), sns.wide(tables["fig3e"], "SNS")
print("Fig. 3E: time 0 of the SNS series equals time 0 of the no-SNS series in every heart:",
      bool((fig3e[0].loc[0] == fig3e[1].loc[0]).all()))

The four CSV files repeat the Fig2 sheet, value for value; each recording is counted once, from the
workbook. No heart's series is repeated in another table. In the Figure 3E table every heart's
"no SNS" APD is the same at 0 s and 5 s (56.9, 56.02, 57.3 ms), and the SNS series starts from those
same values; the first is consistent with a value carried over from 0 s, not a measurement at 5 s.

In [ ]:
# Abstract: mouse APD80 50.9 +- 5.1 ms at 0 s, 60.6 +- 4.1 at 15 s, 46.5 +- 9.1 at 60 s (mean +- SD).
apd = sns.wide(tables["fig2_mouse"], "APD80")
print(sns.mean_sd(apd).loc[[0, 15, 60]].round(2).to_string())
# Fig1 (20 s SNS) against Fig2 time 0: are these the same hearts?
for species in ("rabbit", "mouse"):
    f1 = tables[f"fig1_{species}_hr"]
    f2 = sns.wide(tables[f"fig2_{species}"], "HR")
    print(f"{species}: Fig1 baseline HR {sorted(f1.baseline.tolist())}; Fig2 HR at 0 s {sorted(f2.loc[0].tolist())}")
# Fig5F/7F base and apex change at 15 s against the whole-field APD change at 15 s in Fig2.
for species, key in (("rabbit", "fig5f_rabbit"), ("mouse", "fig7f_mouse")):
    a = sns.wide(tables[f"fig2_{species}"], "APD80")
    print(f"{species}: whole-field APD change at 15 s (Fig2) {sorted((a.loc[15] - a.loc[0]).round(1).tolist())}; "
          f"base {sorted(tables[key].base.tolist())}; apex {sorted(tables[key].apex.tolist())}")

The mouse APD80 table gives 50.52 +- 4.52 ms at 0 s and 61.18 +- 3.24 at 15 s, close to the Abstract's
50.9 +- 5.1 and 60.6 +- 4.1, but 49.54 +- 4.94 at 60 s against the Abstract's 46.5 +- 9.1: the
deposited values are not exactly those the paper summarises. The Figure 1 heart rates are not those
of the Figure 2 hearts at time 0, so Figure 1 and Figure 2 are separate recordings, or the Figure 1
baseline is taken differently. In the mouse every whole-field change in APD at 15 s lies in the range
spanned by the base and apex changes (4.7 to 16.7 ms); in the rabbit three of the four do, and one
(-6.6 ms) is smaller than every base and apex change (-62.9 to -7.4 ms). This is roughly consistent
with Figures 5F and 7F coming from the same experiments as Figure 2, but no identifier links them.

The workbook gives values at stated time points only; there are no traces, frames or event times, so
no clock can be checked between data streams.

Units and labels: Figure 1 has no units for heart rate. The Fig3 tables for tyramine and
noradrenaline label time as "SNS Time (sec)" although these hearts were not stimulated (bolus
infusion; Methods), their time points are not round (6.75, 17.75, 28.5, 59.5 s), and column B of each
holds unlabelled values (0, 3.5, 3.403, 4.435, 9.469 for tyramine) that are consistent with the spread
of the time points across hearts but are not described anywhere.

## fig1a-rabbit-hr-increase and fig1d-mouse-hr-increase

> In innervated rabbit (mouse) hearts, 20 s of SNS increased heart rate over baseline.

The Fig1 sheet gives one baseline and one SNS heart rate per heart. Which window of the 20 s it is
(the claims file's readings: at 20 s, the mean over the 20 s, or the peak) is not stated in the
dataset, so the three readings cannot be told apart: the one value is tested.

In [ ]:
res_fig1 = {}
for species in ("rabbit", "mouse"):
    t = tables[f"fig1_{species}_hr"]
    r = sns.paired(t.baseline, t.sns)
    res_fig1[species] = r
    print(f"{species}: n = {len(t)}; baseline {t.baseline.mean():.1f} +- {t.baseline.std():.1f} bpm, "
          f"SNS {t.sns.mean():.1f} +- {t.sns.std():.1f}; change per heart {(t.sns - t.baseline).tolist()}")
    print(f"   paired t-test, two-sided: t = {r['t']:.2f}, p = {r['p']:.2g}; hearts higher with SNS {(t.sns > t.baseline).sum()} of {len(t)}")

**Result: Reproduced in both species.** Rabbit heart rate rose from 140.2 to 237.2 bpm (all 5 hearts,
paired t-test p = 0.0011) and mouse heart rate from 364.8 to 488.0 bpm (all 5 hearts, p = 0.0033). The window the
SNS value stands for is not stated.

## fig1bc-rabbit-ap-cat-response and fig1ef-mouse-ap-cat-response

> In the rabbit heart SNS shortened APD and CaTD and increased the normalized CaT amplitude.
> In the mouse heart SNS increased heart rate and CaT amplitude and shortened CaTD, but prolonged APD.

The dataset holds no APD, CaTD or CaT amplitude for the 20 s SNS hearts of Figure 1. They are tested on
the first 20 s of the 60 s SNS series of Figure 2 (rabbit APD and CaTD 4 hearts, mouse 5; CaT amplitude
6 hearts each), a deviation from the paper's data. Windows, as the claims file reads "during SNS": (a)
the value at 20 s; (b) the mean of 5, 10, 15 and 20 s; (c) the peak, the most extreme value in the
stated direction from 5 to 20 s. Tests: (1) paired t-test against time 0 (amplitude: against 1, its
value at time 0); (2) the stated direction in the majority of hearts.

In [ ]:
def window_values(w, reading, direction):
    span = w.loc[[5, 10, 15, 20]]
    if reading == "a: at 20 s":
        return w.loc[20]
    if reading == "b: mean 5-20 s":
        return span.mean()
    return span.max() if direction > 0 else span.min()

def direction_table(species, wanted):
    rows = []
    for var, direction in wanted.items():
        key = f"fig2_{species}_cat" if var == "CaT amplitude" else f"fig2_{species}"
        w = sns.wide(tables[key], var)
        for reading in ("a: at 20 s", "b: mean 5-20 s", "c: peak 5-20 s"):
            v = window_values(w, reading, direction)
            r = sns.paired(w.loc[0], v)
            n_dir = int((np.sign(v - w.loc[0]) == direction).sum())
            rows.append({"variable": var, "direction": "up" if direction > 0 else "down", "window": reading,
                         "n": r["n"], "baseline": w.loc[0].mean(), "SNS": v.mean(), "mean change": r["mean_diff"],
                         "t": r["t"], "p": r["p"], "hearts in direction": n_dir,
                         "test holds": (r["p"] < 0.05) and np.sign(r["mean_diff"]) == direction,
                         "majority holds": n_dir > len(v) / 2})
    return pd.DataFrame(rows)

fig1bc = direction_table("rabbit", {"APD80": -1, "CaTD80": -1, "CaT amplitude": 1})
print("Rabbit"); print(fig1bc.round(4).to_string(index=False))

**Result: Reproduced, on the 60 s series.** In the rabbit, under every window, APD80 and CaTD80 are
lower than at time 0 in all 4 hearts and the CaT amplitude is above 1 in all 6 hearts; by paired
t-test the shortening of APD80 at 20 s (-24.6 ms) has p = 0.13 (the mean of 5 to 20 s, p = 0.14),
CaTD80 at 20 s (-39.5 ms) p = 0.020 and the amplitude at 20 s (1.122) p = 0.0006. The claim holds under the
majority reading for all three, and under the paired t-test reading for CaTD80 and amplitude only.
These hearts are those of Figure 2, not Figure 1.

In [ ]:
fig1ef = direction_table("mouse", {"HR": 1, "CaT amplitude": 1, "CaTD80": -1, "APD80": 1})
print("Mouse"); print(fig1ef.round(4).to_string(index=False))

**Result: Reproduced, on the 60 s series.** In the mouse, under every window and both tests, heart rate
and CaT amplitude rise, CaTD80 falls and APD80 rises (APD80 at 20 s: +9.5 ms, 5 of 5 hearts,
p = 0.0002; CaTD80 at 20 s: -3.4 ms, 5 of 5, p = 0.042; amplitude at 20 s: 1.107, 6 of 6, p = 0.0027;
heart rate at 20 s: +60.8 bpm, 5 of 5, p = 0.011).
These hearts are those of Figure 2, not Figure 1.

## fig2a-rabbit-monotonic

> Over 60 s of SNS in the rabbit, heart rate increased monotonically while APD and CaTD decreased
> monotonically, with significant changes from time 0.

Readings of "monotonic": (a) the mean over hearts moves in the stated direction at every successive
time point; (b) no successive pair differs in the opposite direction by paired t-test at p < 0.05.
Significance against time 0 at 60 s: (i) paired t-test; (ii) repeated measures ANOVA, then comparisons
against time 0 with the Holm-Sidak adjustment (the adjustment is this attempt's assumption).

In [ ]:
def time_course_report(long, wanted, label):
    out = {}
    for var, direction in wanted.items():
        w = sns.wide(long, var)
        ms = sns.mean_sd(w)
        mono = sns.monotonic(w, direction)
        tests = sns.against_time0(w)
        out[var] = (w, ms, mono, tests)
        print(f"--- {label}: {var} (expected {({1: 'up', -1: 'down', 0: 'biphasic'})[direction]})")
        print(pd.concat([ms.round(2), tests.round(4)], axis=1).to_string())
        print("  " + sns.anova_line(tests))
        if direction == 0:
            continue
        print(f"  monotonic (a) mean at every step: {mono['a']}; steps against: {[(a, b, round(d, 2)) for a, b, d in mono['a_broken']]}")
        print(f"  monotonic (b) no significant step against: {mono['b']}; significant steps against: "
              f"{[(a, b, round(d, 2), round(p, 4)) for a, b, d, p in mono['b_broken']]}")
    return out

fig2a = time_course_report(tables["fig2_rabbit"], {"HR": 1, "APD80": -1, "CaTD80": -1}, "rabbit")

In [ ]:
summary = []
for var, (w, ms, mono, tests) in fig2a.items():
    d = 1 if var == "HR" else -1
    sig_i = tests.loc[60, "p"] < 0.05 and np.sign(tests.loc[60, "mean_diff"]) == d
    sig_ii = tests.attrs["anova_p"] < 0.05 and tests.loc[60, "p_rm_holm_sidak"] < 0.05 and np.sign(tests.loc[60, "mean_diff"]) == d
    sig_iip = tests.attrs["anova_p"] < 0.05 and tests.loc[60, "p_rm_pooled_holm_sidak"] < 0.05 and np.sign(tests.loc[60, "mean_diff"]) == d
    peak_t = ms["mean"].idxmax() if d > 0 else ms["mean"].idxmin()
    summary.append({"variable": var, "mono (a)": mono["a"], "mono (b)": mono["b"], "60 s sig (i)": sig_i,
                    "60 s sig (ii)": sig_ii, "60 s sig (ii pooled)": sig_iip, "time of extreme mean": peak_t,
                    "change 0 to extreme": ms["mean"][peak_t] - ms["mean"][0],
                    "change 0 to 60 s": ms["mean"][60] - ms["mean"][0]})
summary = pd.DataFrame(summary)
print(summary.round(2).to_string(index=False))
combos = {(m, s): all(summary[f"mono ({m})"] & summary[f"60 s sig ({s})"]) for m in "ab" for s in ("i", "ii", "ii pooled")}
print("all three variables hold under (monotonic, significance) reading:", combos)
# Per heart: does each heart's series move in the stated direction at every step?
for var, (w, *_ ) in fig2a.items():
    d = 1 if var == "HR" else -1
    print(var, "hearts monotonic at every step:", [bool((np.sign(np.diff(w[h].values)) == d).all()) for h in w.columns])

**Result: Not reproduced.** At 60 s every variable
differs from time 0 in the stated direction by paired t-test (HR +65.75 bpm, p = 0.0004; APD80
-31.3 ms, p = 0.021; CaTD80 -33.75 ms, p = 0.0028). After the repeated measures ANOVA, Holm-Sidak
adjusted paired t-tests find HR and CaTD80 at 60 s (p = 0.0015 and 0.0246) but not APD80 (p = 0.18);
Holm-Sidak comparisons on the ANOVA's pooled error, a variant, find all three at 60 s (APD80
p < 0.0001). With the Greenhouse-Geisser correction the APD80 ANOVA is not significant (p = 0.066),
while the HR and CaTD80 ANOVAs are (p = 0.0005 and 0.020). APD80 falls at every step of the mean.
Heart rate is not monotonic: the mean peaks at 20 s and falls by 11.75 bpm to 60 s, and the fall from
50 to 60 s is significant (paired t-test p = 0.047). CaTD80 is monotonic under reading (b) but not
(a): its mean is lowest at 25 s and rises by 5.9 ms to 60 s without a significant step. No
combination of readings has all three variables monotonic and significant at 60 s (the table above),
and the criterion states no partial result, so the claim is not reproduced. Under reading (b) heart
rate fails only on one unadjusted step test of 9, at p = 0.047; were that step not significant,
reading (b) with (i) would hold for all three variables. Under reading (a) heart rate fails whatever
the tests, as 5 of 9 steps of the mean go the wrong way. What holds is the direction of the change
from time 0 at 60 s for all three variables under reading (i), and the monotonic fall of APD80.

## fig2d-mouse-biphasic-apd

> In the mouse, over 60 s of SNS, heart rate increased monotonically and CaTD decreased monotonically,
> but APD was biphasic, prolonging to a maximum near 15 s (50.9 +- 5.1 ms at 0 s, 60.6 +- 4.1 ms at
> 15 s, p < 0.05) and shortening to near control by 60 s (46.5 +- 9.1 ms, not different from 0 s).

In [ ]:
fig2d = time_course_report(tables["fig2_mouse"], {"APD80": 0, "HR": 1, "CaTD80": -1}, "mouse")

In [ ]:
w, ms, _, tests = fig2d["APD80"]
expected = {0: (50.9, 5.1), 15: (60.6, 4.1), 60: (46.5, 9.1)}
for t, (v, tol) in expected.items():
    print(f"APD80 at {t:>2} s: measured {ms['mean'][t]:.2f} ms, expected {v} +- {tol}: within tolerance {abs(ms['mean'][t] - v) <= tol}")
print(f"time of the largest mean APD80: {ms['mean'].idxmax():g} s")
for t in (15, 60):
    print(f"{t} s against 0 s: (i) paired t p = {tests.loc[t, 'p']:.4f}; (ii) RM ANOVA p = {tests.attrs['anova_p']:.2g}, "
          f"Holm-Sidak p = {tests.loc[t, 'p_rm_holm_sidak']:.4f} (pooled error {tests.loc[t, 'p_rm_pooled_holm_sidak']:.4f}); mean change {tests.loc[t, 'mean_diff']:.2f} ms")
print("HR monotonic up (a, b):", fig2d["HR"][2]["a"], fig2d["HR"][2]["b"],
      "; CaTD80 monotonic down (a, b):", fig2d["CaTD80"][2]["a"], fig2d["CaTD80"][2]["b"])
print("per heart, time of the largest APD80:", w.idxmax().tolist())

**Result: Reproduced.** Mean APD80 is 50.52 ms at 0 s, 61.18 at 15 s and 49.54 at 60 s, each within the
stated SD of the paper's value; it peaks at 15 s in the mean and in every heart. 15 s exceeds 0 s
(+10.66 ms; paired t-test p = 0.0002, Holm-Sidak after the repeated measures ANOVA p = 0.0018) and
60 s does not differ from 0 s (-0.98 ms; p = 0.47 and 0.72). The pooled-error Holm-Sidak variant agrees
(15 s p < 0.0001, 60 s p = 0.51). Heart rate rises and CaTD80 falls at
every step of the mean. The 60 s SD (4.94 ms) is about half of the Abstract's 9.1.

## fig2cf-cat-amplitude-similar

> Both species increased relative CaT amplitude during 60 s of SNS, with similar magnitude and time
> course.

Increase: relative amplitude above 1 at p < 0.05 against time 0 (paired t-test, equal to a one-sample
t-test against 1) at some time point. Similarity: (a) at each sampled time after 0 each species' mean
within the other's SD; (b) unpaired two-sided t-test between species at each time, p >= 0.05. Time 0 is
1 by construction in both and is left out of the similarity readings.

In [ ]:
rab = sns.wide(tables["fig2_rabbit_cat"], "CaT amplitude")
mou = sns.wide(tables["fig2_mouse_cat"], "CaT amplitude")
rows = []
for t in rab.index:
    r_r, r_m = sns.paired(rab.loc[0], rab.loc[t]), sns.paired(mou.loc[0], mou.loc[t])
    u = stats.ttest_ind(rab.loc[t], mou.loc[t]) if t > 0 else None
    rows.append({"time": t, "rabbit mean": rab.loc[t].mean(), "rabbit sd": rab.loc[t].std(), "rabbit p vs 0": r_r["p"],
                 "mouse mean": mou.loc[t].mean(), "mouse sd": mou.loc[t].std(), "mouse p vs 0": r_m["p"],
                 "means within other's SD": (abs(rab.loc[t].mean() - mou.loc[t].mean()) <= min(rab.loc[t].std(), mou.loc[t].std())) if t > 0 else None,
                 "unpaired p": u.pvalue if u is not None else None})
cat = pd.DataFrame(rows).set_index("time")
print(cat.round(4).to_string())
for name, w in (("rabbit", rab), ("mouse", mou)):
    tests = sns.against_time0(w)
    print(f"{name}: {sns.anova_line(tests)}; Holm-Sidak p against time 0: {tests['p_rm_holm_sidak'].round(4).tolist()}")
c = cat.loc[cat.index > 0]
print("increase in rabbit:", bool(((c['rabbit p vs 0'] < 0.05) & (c['rabbit mean'] > 1)).any()),
      "; in mouse:", bool(((c['mouse p vs 0'] < 0.05) & (c['mouse mean'] > 1)).any()))
print("similar (a) at every time:", bool(c["means within other's SD"].all()), "; (b) at every time:", bool((c["unpaired p"] >= 0.05).all()))
print("time of the largest mean: rabbit", cat["rabbit mean"].idxmax(), "s, mouse", cat["mouse mean"].idxmax(), "s")

**Result: Reproduced, under similarity reading (b).** The relative amplitude rises above 1 in both
species (rabbit 1.122 at 20 s, p = 0.0006; mouse 1.107 at 20 s, p = 0.0027; by paired t-test the rabbit
is above 1 at every time from 5 s, the mouse from 10 to 30 s). The unpaired t-tests between species give
p from 0.16 to 0.56 at every time point (reading b). Under reading (a) similarity fails at 5, 10 and
15 s, where the rabbit mean is higher than the mouse mean by more than the smaller SD (at 15 s, 1.098
against 1.072, mouse SD 0.018); from 20 s on it holds. The rabbit mean peaks at 30 s (1.123) and the
mouse mean at 20 s (1.107).

## fig3abc-tyramine-na-mimic-sns

> In traditionally perfused mouse hearts, bolus tyramine (20 uM) or noradrenaline (50 nM) produced APD,
> CaTD and HR responses similar to those of SNS.

Read as the pattern of fig2d-mouse-biphasic-apd: HR rising and CaTD falling (monotonic as read in
fig2a, readings a and b), APD above time 0 at an early time point by paired t-test at p < 0.05, and
APD back toward time 0 at the last time point (the mean at the last time below the peak mean). The
time points are those of the table (mean times across hearts; the bolus times are not round).

In [ ]:
fig3 = {}
for agent, key in (("tyramine", "fig3_tyramine"), ("noradrenaline", "fig3_na")):
    out = time_course_report(tables[key], {"HR": 1, "CaTD80": -1, "APD80": 1}, agent)
    w, ms, _, tests = out["APD80"]
    early = tests.index[0]
    t_last = w.index.max()
    hr, catd = out["HR"][2], out["CaTD80"][2]
    pattern = {
        "HR rising (a)": hr["a"], "HR rising (b)": hr["b"], "CaTD falling (a)": catd["a"], "CaTD falling (b)": catd["b"],
        "APD up at an early time (p<0.05)": bool(((tests.p < 0.05) & (tests.mean_diff > 0) & (tests.index <= 20)).any()),
        "peak APD time": ms["mean"].idxmax(),
        "APD last below peak": bool(ms["mean"][t_last] < ms["mean"].max()),
        "APD last minus time 0": round(ms["mean"][t_last] - ms["mean"][0], 2),
        "APD last vs 0 p": round(tests.loc[t_last, "p"], 4)}
    fig3[agent] = pattern
    print(agent, pattern, "\n")

In [ ]:
ok = {}
for agent, p in fig3.items():
    ok[agent] = {"a": p["HR rising (a)"] and p["CaTD falling (a)"], "b": p["HR rising (b)"] and p["CaTD falling (b)"]}
    ok[agent] = {k: v and p["APD up at an early time (p<0.05)"] and p["APD last below peak"] for k, v in ok[agent].items()}
print("pattern holds, by monotonic reading:", ok)

**Result: Reproduced, for noradrenaline under monotonic reading (b) only.** Tyramine reproduces the SNS pattern under both readings: heart rate
rises and CaTD80 falls at every step of the mean, APD80 is above time 0 from 6.75 s (+8.38 ms,
p = 0.017), peaks at 17.75 s and returns toward time 0 by 59.5 s (+2.13 ms, p = 0.085). Noradrenaline
gives the APD response (above time 0 at 6.25 s, +9.53 ms, p = 0.010; peak at 12.33 s; 3.20 ms below
time 0 at 59.75 s, p = 0.046) and CaTD80 falls at every step, but its heart rate falls from 29.75 to
59.75 s in all 4 hearts (mean -26.75 bpm), so heart rate is not monotonic under reading (a); under
reading (b) that fall is not significant at p < 0.05 and the pattern holds. One noradrenaline heart
has no value at 12.33 s. The claim holds for both agents under reading
(b) and for tyramine alone under reading (a). The criterion asks for heart rate rising, and the
noradrenaline heart rate stays above time 0 throughout, so the outcome is reproduced. Reading (b) has little power with 3 or 4 hearts: the
noradrenaline heart rate falls by 26.75 bpm in the mean from 29.75 to 59.75 s, in all 4 hearts,
though it stays 82.5 bpm above time 0 (p = 0.023).

## fig3d-propranolol-abolishes

> Propranolol (10 uM) completely abolished the HR, APD and CaTD effects of SNS in the mouse heart.

A claim of absence. The paper's test, repeated measures ANOVA or paired t-tests against time 0 at
p < 0.05, is run both ways on n = 3. Reproduced when no time point of HR, APD80 or CaTD80 differs from
time 0 under either test.

In [ ]:
prop = time_course_report(tables["fig3_propranolol"], {"HR": 1, "APD80": 1, "CaTD80": -1}, "propranolol + SNS")
rows = []
for var, (w, ms, _, tests) in prop.items():
    biggest = tests.mean_diff.abs().idxmax()
    rows.append({"variable": var, "min paired p": tests.p.min(), "at": tests.p.idxmin(),
                 "RM ANOVA p": tests.attrs["anova_p"], "RM ANOVA p (GG)": tests.attrs["anova_p_gg"],
                 "min Holm-Sidak p": tests.p_rm_holm_sidak.min(),
                 "min pooled Holm-Sidak p": tests.p_rm_pooled_holm_sidak.min(),
                 "at (pooled)": tests.p_rm_pooled_holm_sidak.idxmin(),
                 # comparisons after the ANOVA follow only a significant ANOVA, as in fig2a
                 "pooled p after ANOVA p < 0.05": tests.p_rm_pooled_holm_sidak.min() if tests.attrs["anova_p"] < 0.05 else np.nan,
                 "largest mean change": tests.mean_diff[biggest], "at time": biggest,
                 "relative to time 0 (%)": 100 * tests.mean_diff[biggest] / ms["mean"][0]})
prop_sum = pd.DataFrame(rows)
print(prop_sum.round(4).to_string(index=False))
# SNS without propranolol (Fig. 2D), largest mean change, for scale.
for var in ("HR", "APD80", "CaTD80"):
    w = sns.wide(tables["fig2_mouse"], var)
    d = w.sub(w.loc[0]).mean(axis=1)
    print(f"Fig2 mouse SNS without propranolol, {var}: largest mean change {d[d.abs().idxmax()]:.1f} at {d.abs().idxmax():g} s")
n_tests = sum(len(v[3]) for v in prop.values())
print(f"unadjusted paired t-tests against time 0: {n_tests}; if independent and with no effect, the chance that at least "
      f"one reaches p < 0.05 is {1 - 0.95 ** n_tests:.2f}")
w = prop["APD80"][0]
print("APD80 per heart, change from time 0 at 10 s:", (w.loc[10] - w.loc[0]).round(2).tolist())
w = prop["HR"][0]
print("HR per heart (bpm) at 0 s:", w.loc[0].tolist(), "; at 60 s:", w.loc[60].tolist(),
      "; change:", (w.loc[60] - w.loc[0]).tolist())


**Result: Not reproduced.** By paired t-tests against time 0 (unadjusted, reading (i)), APD80 differs
at 10 s (+1.43 ms, p = 0.010), and the outcome rests on this test under the claims file's rule (either
test, any time point). Its per-heart changes are 1.2, 1.4 and 1.7 ms. It is one of 27 unadjusted tests
on 3 hearts, among which at least one p < 0.05 would arise by chance with probability 0.75 if the tests
were independent and there were no effect. The repeated measures ANOVA is significant for heart rate
(p = 0.031), not for APD80 (p = 0.11) or CaTD80 (p = 0.28); with the Greenhouse-Geisser correction none
is significant (heart rate p = 0.23). After the ANOVA no Holm-Sidak adjusted paired t-test against time
0 reaches p < 0.05 (smallest 0.088, APD80 at 10 s). Holm-Sidak comparisons on the ANOVA's pooled error,
a variant, are counted only after a significant ANOVA, so only for heart rate; they find it above time
0 at 30, 40, 50 and 60 s (smallest p = 0.0048, at 60 s, +16.7 bpm). This rests on the uncorrected ANOVA,
which assumes sphericity, and on 3 hearts, of which heart rate rises from time 0 to 60 s in 2 (+32 and
+20 bpm) and falls in 1 (-2 bpm). The changes are small: the largest mean changes are +16.7 bpm in heart
rate (+6.2%), +3.5 ms in APD80 (+9.4%) and -0.73 ms in CaTD80, against +87.4 bpm, +10.7 ms and -7.2 ms
with SNS alone (Figure 2D table).

## fig3e-rate-step-biphasic

> A step-wise increase in pacing rate mimicking the SNS heart rate change produced a biphasic APD
> response without SNS, to a lesser degree, and adding SNS prolonged APD and exaggerated the biphasic
> response.

Biphasic: a rise above time 0 followed by a fall below the peak by the end of the record. The table
ends at 35 s (the claims file says 60 s; the record's last point is used). Exaggerated: (a) a larger peak
rise of the mean above time 0 with SNS; (b) APD with SNS above APD without SNS at the time of the peak,
by paired t-test at p < 0.05 (the peak time of each condition's mean is shown).

In [ ]:
e0, e1 = sns.wide(tables["fig3e"], "no SNS"), sns.wide(tables["fig3e"], "SNS")
print("pacing rate (bpm):", sns.wide(tables["fig3e"], "pacing rate (bpm)").iloc[:, 0].round(1).tolist())
res = {}
for name, w in (("no SNS", e0), ("SNS", e1)):
    ms = sns.mean_sd(w)
    tests = sns.against_time0(w)
    peak_t = ms["mean"].idxmax()
    res[name] = {"peak time": peak_t, "peak rise": ms["mean"][peak_t] - ms["mean"][0],
                 "end minus peak": ms["mean"].iloc[-1] - ms["mean"][peak_t],
                 "biphasic": bool(ms["mean"][peak_t] > ms["mean"][0] and ms["mean"].iloc[-1] < ms["mean"][peak_t]),
                 "end minus time 0": ms["mean"].iloc[-1] - ms["mean"][0]}
    print(f"--- {name}"); print(pd.concat([ms.round(2), tests.round(4)], axis=1).to_string()); print("  " + sns.anova_line(tests))
print(pd.DataFrame(res).T.to_string())
for name in res:
    t = res[name]["peak time"]
    r = sns.paired(e0.loc[t], e1.loc[t])
    print(f"SNS minus no SNS at {t:g} s (peak of {name}): {r['mean_diff']:.2f} ms, paired t = {r['t']:.2f}, p = {r['p']:.4f}")
print("exaggerated (a):", res["SNS"]["peak rise"] > res["no SNS"]["peak rise"])
print("per heart, peak rise without / with SNS:", (e0.max() - e0.loc[0]).round(2).tolist(), (e1.max() - e1.loc[0]).round(2).tolist())

**Result: Reproduced, under reading (a).** Both conditions are biphasic: without SNS the mean APD80
rises from 56.74 to 66.02 ms at 10 s (+9.28) and falls to 53.83 at 35 s; with SNS it rises to 71.57 ms
at 10 s (+14.83) and falls to 57.27 at 35 s. The peak rise is larger with SNS in the mean (reading a)
and in each of the 3 hearts. By reading (b) SNS is above no SNS at 10 s by 5.55 ms, p = 0.13, so the
claim holds under (a) only. The record ends at 35 s, not 60 s.

## fig3f-constant-pacing-sns-prolongs

> In the mouse at constant pacing (CL 100 ms), SNS eliminated the biphasic APD response and APD was
> prolonged compared to baseline; APD tended to shorten over 60 s of pacing both without and with SNS.

Prolonged compared to baseline: (a) APD with SNS above APD without SNS at the same time point by paired
t-test at p < 0.05 at some point; (b) APD with SNS above its own time 0 at p < 0.05 at some point.
Tended to shorten: mean at 60 s below the mean at the first time, in both. No biphasic response with
SNS: the SNS mean does not rise above time 0 and then fall. The no-SNS series has 4 hearts (column 2 of
its block is empty); pairing with the SNS series assumes the columns are the same hearts in the same
order, which the table does not state. An unpaired Welch t-test over all hearts is shown as a variant.

In [ ]:
f0, f1 = sns.wide(tables["fig3f"], "no SNS"), sns.wide(tables["fig3f"], "SNS")
print("no-SNS hearts (column positions):", f0.columns.tolist(), "; SNS hearts:", f1.columns.tolist())
rows = []
for t in f1.index:
    r = sns.paired(f0.loc[t].reindex(f1.columns), f1.loc[t])
    welch = stats.ttest_ind(f1.loc[t], f0.loc[t].dropna(), equal_var=False)
    rows.append({"time": t, "no SNS mean": f0.loc[t].mean(), "SNS mean": f1.loc[t].mean(),
                 "paired n": r["n"], "SNS minus no SNS (paired)": r["mean_diff"], "paired p": r["p"],
                 "Welch p": welch.pvalue})
fig3f = pd.DataFrame(rows).set_index("time")
print(fig3f.round(4).to_string())
for name, w in (("no SNS", f0), ("SNS", f1)):
    tests = sns.against_time0(w)
    ms = sns.mean_sd(w)
    print(f"--- {name}: {sns.anova_line(tests)}")
    print(pd.concat([ms.round(2), tests.round(4)], axis=1).to_string())
    print(f"  60 s minus 0 s: {ms['mean'][60] - ms['mean'][0]:.2f} ms; largest mean at {ms['mean'].idxmax():g} s; "
          f"any time above time 0 at p<0.05: {bool(((tests.p < 0.05) & (tests.mean_diff > 0)).any())}")
sns_ms = sns.mean_sd(f1)["mean"]
biphasic_sns = bool(sns_ms.max() > sns_ms[0] and sns_ms.iloc[-1] < sns_ms.max())
print("SNS mean rises above time 0 and then falls (biphasic):", biphasic_sns)
print("reading (a) holds:", bool(((fig3f["paired p"] < 0.05) & (fig3f["SNS minus no SNS (paired)"] > 0)).any()),
      "; variant, Welch:", bool(((fig3f["Welch p"] < 0.05) & (fig3f["SNS mean"] > fig3f["no SNS mean"])).any()))
# The pairing by column position. Per heart, the gap at time 0, and how close the paired series are
# against every other assignment of the 4 no-SNS columns to 4 of the 5 SNS columns.
import itertools
print("SNS minus no SNS at time 0, per column:", (f1.loc[0] - f0.loc[0]).dropna().round(2).to_dict())
def mismatch(assign):
    return sum(float(((f1[s] - f0[b]) ** 2).sum()) for b, s in assign.items())
by_position = mismatch({b: b for b in f0.columns})
others = [mismatch(dict(zip(f0.columns, perm))) for perm in itertools.permutations(f1.columns, len(f0.columns))]
print(f"sum of squared differences over the 60 s, pairing by position: {by_position:.1f}; rank among all "
      f"{len(others)} assignments: {sum(o < by_position for o in others) + 1}; smallest {min(others):.1f}")
# Variant: change from time 0, SNS against no SNS (removes the gap present at time 0).
c0, c1 = f0.sub(f0.loc[0]), f1.sub(f1.loc[0])
rows = []
for t in f1.index[1:]:
    r = sns.paired(c0.loc[t].reindex(f1.columns), c1.loc[t])
    rows.append({"time": t, "change no SNS": c0.loc[t].mean(), "change SNS": c1.loc[t].mean(),
                 "SNS minus no SNS (paired)": r["mean_diff"], "paired p": r["p"]})
print(pd.DataFrame(rows).set_index("time").round(4).to_string())


**Result: Reproduced, under reading (a) and resting on the column pairing.** With SNS the mean APD80 is
above the no-SNS mean at every time point (by 0.5 to 3.9 ms on the paired hearts) and the paired
t-test reaches p < 0.05 at 0, 5, 10, 15 and 20 s (smallest p = 0.0023, at 15 s), on 4 hearts paired by
column position. APD80 shortens over 60 s in both conditions (no SNS by 11.55 ms, SNS by 13.78 ms), and
the SNS mean never rises above its time 0, so there is no biphasic response. Reading (b) fails: with
SNS, APD80 is never above its own time 0. The unpaired Welch variant finds no time point at p < 0.05.
The SNS series is already longer than the no-SNS series at time 0, in each of the 4 pairs (by 1.65 to
4.84 ms), so the difference is present at the onset of SNS. On the change from time 0, SNS and no
SNS do not differ at any time (paired p from 0.18 to 0.94). The pairing by column position is the 4th
closest of the 120 ways of assigning the 4 no-SNS columns to the 5 SNS columns (sum of squared
differences 400.9 against a smallest 380.3), consistent with, but not proof of, the columns being the
same hearts. The outcome rests on that pairing and on reading (a), which compares the two series at
the same times without removing the gap at time 0.

## fig4g-5f-rabbit-base-apex and fig7f-mouse-base-apex

> In the rabbit, SNS shortened APD more at the base than at the apex ... at 15 s SNS the APD change
> differed between base and apex.
> In the mouse, APD prolonged more at the base than at the apex at 15 s SNS.

The workbook gives, per heart, the change in APD at 15 s of SNS in the base and apex regions as the
authors drew them (Figure 5E and 7E). The claims file's readings of the regions (basal and apical
thirds or quarters of the map) cannot be applied, as there are no maps; the authors' own regions are
used. Paired t-test, base against apex.

In [ ]:
for key, claim, direction, alpha in (("fig5f_rabbit", "rabbit", -1, 0.05), ("fig7f_mouse", "mouse", 1, 0.01)):
    t = tables[key]
    r = sns.paired(t.apex, t.base)
    print(f"{claim}: n = {len(t)}; base {t.base.mean():.2f} +- {t.base.std():.2f} ms, apex {t.apex.mean():.2f} +- {t.apex.std():.2f}; "
          f"base minus apex per heart {(t.base - t.apex).round(1).tolist()}")
    print(f"   paired t-test: mean base minus apex {r['mean_diff']:.2f} ms, t = {r['t']:.2f}, p = {r['p']:.4f}; "
          f"base change larger in the stated direction in {int((np.sign(t.base - t.apex) == direction).sum())} of {len(t)}; "
          f"holds at p < {alpha}: {bool(r['p'] < alpha and np.sign(r['mean_diff']) == direction)}")
    lo = t.drop(t.index[(t.base - t.apex).abs().idxmax()])
    rl = sns.paired(lo.apex, lo.base)
    print(f"   without the heart of largest difference: p = {rl['p']:.4f}; Wilcoxon signed-rank on all: "
          f"p = {stats.wilcoxon(t.base, t.apex).pvalue:.4f}")

**Result: Rabbit reproduced; mouse reproduced.** In the rabbit the base shortened more than the apex in
all 4 hearts (base -28.40 ms, apex -18.25; mean difference -10.15 ms, paired t-test p = 0.048). Without
the heart of largest difference (heart 3, -62.9 against -43.7 ms) p = 0.029; the Wilcoxon signed-rank
test, whose smallest possible p with 4 hearts is 0.125, gives 0.125. In the mouse the base
prolonged more than the apex in all 5 hearts (base +14.46 ms, apex +7.04; difference +7.42 ms,
p = 0.0019, below 0.01). That the base-apex difference causes the repolarization reversal is not
testable here; the maps are not in the dataset.

## Simulation claims (fig8e, fig8f, fig8ij, fig9)

The claims file marks these as data elsewhere, at the Bers and Grandi lab pages. The dataset itself holds
both models in `files/code/mouse` and `files/code/rabbit`, with the AP-clamp voltage inputs and
instructions for Figure 8. They are printed here, with a check for MATLAB and GNU Octave on the machine
that runs the notebook.

In [ ]:
import shutil
print(files[files.path.str.startswith("files/code")].sort_values("path").to_string(index=False))
for path in ["files/code/mouse/instructions_mouse.txt", "files/code/rabbit/instructions_rabbit.txt"]:
    print(f"--- {path}")
    print(sns.read_file(man, path).decode("utf8", "replace").strip())
print("matlab on PATH:", shutil.which("matlab"), "; octave on PATH:", shutil.which("octave"))


**Result: Blocked, not runnable.** The models and the instructions to reproduce Figure 8 are in the
dataset, but they are MATLAB programs (`ode15s` solvers over six files per species). Neither MATLAB nor
GNU Octave is on the machine that runs the notebook (the check above). GNU Octave can be installed
without root from conda-forge, but the entry's environment is a Python virtual environment built from a
pip lock file, which cannot hold it, so the simulations cannot be run as part of this entry. A trial
outside the notebook, not part of the entry and not deciding the outcome, found that the mouse model
starts under Octave after a one-line change to its `ode15s` call and runs far slower than real time, so
its 70 s simulations would likely not fit the notebook's time limit. The instructions cover Figure 8
(AP-clamp with APD fixed, mouse-like or rabbit-like, and current clamp); they do not name Figure 9.

## Summary

| Claim | Outcome |
|---|---|
| fig1a-rabbit-hr-increase | Reproduced |
| fig1d-mouse-hr-increase | Reproduced |
| fig1bc-rabbit-ap-cat-response | Reproduced, on the first 20 s of the 60 s series |
| fig1ef-mouse-ap-cat-response | Reproduced, on the first 20 s of the 60 s series |
| fig2a-rabbit-monotonic | Not reproduced: HR is not monotonic under either reading, CaTD80 not under reading (a); under reading (b) this rests on one unadjusted step test of HR (p = 0.047) |
| fig2d-mouse-biphasic-apd | Reproduced |
| fig2cf-cat-amplitude-similar | Reproduced (similarity reading b) |
| fig3abc-tyramine-na-mimic-sns | Reproduced; for noradrenaline only under monotonic reading (b), as its HR falls after 30 s |
| fig3d-propranolol-abolishes | Not reproduced: APD80 at 10 s (+1.4 ms) at p = 0.010 unadjusted; HR at 30 to 60 s with pooled-error Holm-Sidak comparisons after an uncorrected ANOVA (2 of 3 hearts rise) |
| fig3e-rate-step-biphasic | Reproduced (reading a) |
| fig3f-constant-pacing-sns-prolongs | Reproduced (reading a), resting on the column pairing |
| fig4g-5f-rabbit-base-apex | Reproduced |
| fig7f-mouse-base-apex | Reproduced |
| fig4ab, fig4cd, fig4e, fig5c, fig6ab, fig6e, fig6fh, fig7c | Blocked: data missing (no maps or paced series) |
| fig8e, fig8f, fig8ij, fig9 | Blocked: not runnable (MATLAB models, in the dataset; the entry's Python environment cannot hold MATLAB or Octave) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The analysis starts from the authors' per-heart summary
values; the optical mapping analysis (Optiq) is not repeated. The Figure 1 APD, CaTD and amplitude
claims are tested on the first 20 s of the 60 s series. The comparisons after a repeated measures
ANOVA use the Holm-Sidak adjustment, which the paper does not name, on paired t-tests and, as a
variant, on the ANOVA's pooled error; neither corrects for sphericity. The ANOVA's p with the
Greenhouse-Geisser correction is printed beside it and decides no outcome. The Figure 3E record ends at 35 s.
The Figure 3F series are paired by column position, an assumption the fig3f outcome rests on.

**What was found about the dataset.** It holds per-heart summary tables for Figures 1 (heart rate only),
2, 3, 5F and 7F, and the model code and instructions for Figure 8 (`files/code`), but no optical maps or traces, so the
spatio-temporal claims (Figures 4 to 7) cannot be tested. Its metadata declare 0 subjects and call it
a computational modelling dataset; no table identifies an animal. The four CSV files duplicate the
Fig2 sheet. The mouse APD80 at 60 s is 49.54 +- 4.94 ms in the table and 46.5 +- 9.1 in the Abstract.
The Figure 3F baseline block has 4 hearts where the legend says 5. The Figure 3E no-SNS values at 5 s
repeat those at 0 s. The tyramine and noradrenaline tables call their time "SNS Time" and carry an
unlabelled column; Figure 1 heart rates have no unit.